In [1]:
import numpy as np
import pandas as pd

In [2]:

arrays = [
    ["East", "East", "East", "West", "West", "West"],
    ["Store A", "Store B", "Store C", "Store A", "Store B", "Store C"]
]

index = pd.MultiIndex.from_arrays(arrays, names=["Region", "Store"])

df = pd.DataFrame({
    "Sales": [1200, 900, 1100, 1500, 1300, 1250],
    "Employees": [10, 7, 8, 12, 9, 10]
}, index=index)

df

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
       Store C   1100          8
West   Store A   1500         12
       Store B   1300          9
       Store C   1250         10

# Useful operations you can try immediately

## Select all stores in the East region

In [3]:
df.loc['East']

,Sales,Employees
Store,,
Store A,1200,10
Store B,900,7
Store C,1100,8


## Select 'Store B' in all regions

In [6]:
df.xs('Store B', level='Store')

,Sales,Employees
Region,,
East,900,7
West,1300,9


## Select a single row

In [7]:
df.loc[("West", "Store C")]

Sales        1250
Employees      10
Name: (West, Store C), dtype: int64

## Sort by index levels

In [8]:
df.sort_index(level=["Region", "Store"], ascending=[True, False])

Sales  Employees
Region Store                    
East   Store C   1100          8
       Store B    900          7
       Store A   1200         10
West   Store C   1250         10
       Store B   1300          9
       Store A   1500         12

## Reset the Multiindex

In [9]:
df.reset_index()

,Region,Store,Sales,Employees
0,East,Store A,1200,10
1,East,Store B,900,7
2,East,Store C,1100,8
3,West,Store A,1500,12
4,West,Store B,1300,9
5,West,Store C,1250,10


In [10]:
df

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
       Store C   1100          8
West   Store A   1500         12
       Store B   1300          9
       Store C   1250         10

# Bonus: MultiIndex columns example (also common)

In [11]:
df2 = pd.DataFrame({
    ("Sales", "Q1"): [100, 120, 90],
    ("Sales", "Q2"): [110, 130, 95],
    ("Costs", "Q1"): [60, 70, 55],
    ("Costs", "Q2"): [65, 75, 58],
}, index=["Product A", "Product B", "Product C"])

df2.columns = pd.MultiIndex.from_tuples(df2.columns)
df2

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75
Product C    90   95    55  58

# What is df.xs()?

**df.xs()** is one of those pandas methods that feels mysterious until you see what it’s actually doing. After that, it becomes one of the cleanest ways to slice **across levels of a MultiIndex**.

You’ve already been working with MultiIndex DataFrames, so this will click quickly.

The pandas.DataFrame.xs() method is used to return a cross-section (specific rows or columns) from a Series or DataFrame based on a key. 
It is especially useful for selecting data from a MultiIndex. 

**Function Signature**
The format of the method is as follows:

```
DataFrame.xs(key, axis=0, level=None, drop_level=True)
```


### Parameters
- **key** (label or tuple of labels): The label (or partial label in a MultiIndex) used to select data.
- **axis** ({0 or 'index', 1 or 'columns'}, default 0): The axis to retrieve the cross-section on. axis=0 is for rows (index), and axis=1 is for columns.
- **level** (object, default None): In the case of a MultiIndex, this specifies which level(s) the key belongs to. It can be a level name or position.
- **drop_level** (bool, default True): If False, the level(s) selected with the key will remain in the result's index/columns. 

### Returns
**Series** or **DataFrame**: A cross-section from the original object corresponding to the selected index levels. 

### Key Usage Notes
- xs() **cannot be used to set values.**
- It is often used when working with hierarchical indices (MultiIndex) to easily select data at a particular level.
- While **loc** provides more generic indexing capabilities, **xs()** is a convenient way to select a single key across a dimension, especially with multi-indexing. 

## What df.xs() does

It selects a **cross‑section** of the DataFrame by choosing values from **one or more index levels**.

Think of it as:
**“Give me all rows where level X equals value Y.”**

It’s especially useful when you want to slice by a lower index level without writing long tuples.



## Example DataFrame


In [12]:
import pandas as pd

arrays = [
    ["East", "East", "East", "West", "West", "West"],
    ["Store A", "Store B", "Store C", "Store A", "Store B", "Store C"]
]

index = pd.MultiIndex.from_arrays(arrays, names=["Region", "Store"])

df = pd.DataFrame({
    "Sales": [1200, 900, 1100, 1500, 1300, 1250],
    "Employees": [10, 7, 8, 12, 9, 10]
}, index=index)

df

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
       Store C   1100          8
West   Store A   1500         12
       Store B   1300          9
       Store C   1250         10

## 1. Select all rows where Store == "Store B"

In [13]:
df.xs('Store B', level='Store')

,Sales,Employees
Region,,
East,900,7
West,1300,9


This is the most common use: slicing by a lower level without writing df.loc[("East", "Store B")] and df.loc[("West", "Store B")].

## 2. Select all rows where Region== "East"

In [14]:
df.xs("East", level="Region")

,Sales,Employees
Store,,
Store A,1200,10
Store B,900,7
Store C,1100,8


Equivalent to df.loc["East"], but works even when the level you want isn’t the first one.

## 3. Select using multiple levels at once


In [16]:
df.xs(("West", "Store C"), level=["Region", "Store"])

,,Sales,Employees
Region,Store,,
West,Store C,1250,10


This is a clean alternative to ```df.loc[("West", "Store C")]```.

## 4. Select columns with MultiIndex (bonus)

If columns also have multiple levels:


In [17]:
df2

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75
Product C    90   95    55  58

In [21]:
# df2.xs("Q1", level="Quarter", axis=1)

## 5. Using drop_level=False to keep the level in the result

By default, **xs()** removes the level you sliced on.
You can keep it:


In [22]:
df.xs("Store B", level="Store", drop_level=False)

,,Sales,Employees
Region,Store,,
East,Store B,900,7
West,Store B,1300,9


Now the result still has both index levels.

## When to use xs() instead of .loc

Use .loc when you know the full index tuple:


In [23]:
df.loc[("East", "Store A")]

Sales        1200
Employees      10
Name: (East, Store A), dtype: int64

Use .xs when you want:
- all rows matching a value in a specific level
- to slice by a lower level without writing tuples
- to slice by multiple levels cleanly
- to slice MultiIndex columns


Stacking is one of those pandas operations that feels abstract until you see it reshape a real MultiIndex DataFrame. Once it clicks, you’ll recognize it as a powerful way to pivot wide data into long form, especially when your columns have multiple levels.
You already work comfortably with MultiIndexes, so I’ll show you examples that build intuition rather than just syntax.

# 🟦 What stack() does

It moves one column level into the row index, creating a longer, narrower DataFrame (or Series).
If your DataFrame has MultiIndex columns, stacking collapses one level downward.
If your DataFrame has MultiIndex rows, stacking works normally but is less common.

## 🟩 Example 1 — Stacking MultiIndex columns (the classic use case)
Start with a DataFrame that has two column levels:


In [24]:
import pandas as pd

df = pd.DataFrame({
    ("Sales", "Q1"): [100, 120, 90],
    ("Sales", "Q2"): [110, 130, 95],
    ("Costs", "Q1"): [60, 70, 55],
    ("Costs", "Q2"): [65, 75, 58],
}, index=["Product A", "Product B", "Product C"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75
Product C    90   95    55  58

## ▶️ Stack the inner column level (Q1, Q2)


In [25]:
df.stack()

C:\Users\lorra\AppData\Local\Temp\ipykernel_89916\3977694045.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack()


Sales  Costs
Product A Q1    100     60
          Q2    110     65
Product B Q1    120     70
          Q2    130     75
Product C Q1     90     55
          Q2     95     58

What happened:
- The column level Q1/Q2 moved into the row index.
- The DataFrame became longer and narrower.
- The remaining column level is "Sales" and "Costs".
This is the most common real‑world use of stack().

## 🟩 Example 2 — Stack a specific level

If your columns have more than two levels, you can choose which one to stack:

In [27]:
# df.stack(level="Quarter")

## 🟩 Example 3 — Stacking a MultiIndex row DataFrame
This is less common but still useful.


In [28]:
arrays = [
    ["East", "East", "West", "West"],
    ["Store A", "Store B", "Store A", "Store B"]
]

index = pd.MultiIndex.from_arrays(arrays, names=["Region", "Store"])

df2 = pd.DataFrame({
    "Sales": [1200, 900, 1500, 1300],
    "Employees": [10, 7, 12, 9]
}, index=index)

df2

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
West   Store A   1500         12
       Store B   1300          9

### Stack the columns into a new index level

In [29]:
df2.stack()

Region  Store             
East    Store A  Sales        1200
                 Employees      10
        Store B  Sales         900
                 Employees       7
West    Store A  Sales        1500
                 Employees      12
        Store B  Sales        1300
                 Employees       9
dtype: int64

Now the DataFrame becomes a **Series** with a 3‑level index:
- Region
- Store
- Variable (Sales/Employees)

This is the long‑form representation of the original table.

## 🟩 Example 4 — Unstacking back (the inverse)
df2.stack().unstack()





In [31]:
df2

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
West   Store A   1500         12
       Store B   1300          9

In [30]:
df2.stack().unstack()

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
West   Store A   1500         12
       Store B   1300          9

## 🟦 When to use stack()

It shines when:
- You have MultiIndex columns and want tidy/long format
- You want to prepare data for plotting or groupby operations
- You want to normalize wide business reports into row‑based records
- You want to invert a pivot operation


**stack()** is not limited to MultiIndex columns. It shines with MultiIndex columns, but it also works perfectly on regular DataFrames and MultiIndex rows. The key is understanding what it always does:
stack() takes one column level and pushes it into the row index.
If there is only one column level, it simply becomes a new index level.
Let’s walk through the full picture so you can see exactly when and why you’d use it.

## 1. stack() on a regular (single‑level) DataFrame

Even without MultiIndex columns, stack() works.


In [32]:
import pandas as pd

df = pd.DataFrame({
    "Math": [90, 80],
    "Science": [85, 95]
}, index=["Alice", "Bob"])

df

,Math,Science
Alice,90,85
Bob,80,95


In [33]:
# Stack It
df.stack()

Alice  Math       90
       Science    85
Bob    Math       80
       Science    95
dtype: int64

Result (a Series with a 2‑level index)
Even though the columns weren’t a MultiIndex, stacking still works — it just creates a new index level.

## 2. stack() on a DataFrame with MultiIndex columns
This is the most common use case because stacking naturally collapses one column level.
